# Эволюция схемы в Apache Iceberg — на Trino

Демонстрация **гайда 3-ДОП**: меняем схему таблицы Iceberg **без перезаписи данных**.
Вместо Spark — **Trino** через Python-клиент `trino`, прямо из JupyterHub.

Показываем на живом примере:
- какие **типы данных** бывают (примитивы + сложные);
- как **добавлять / переименовывать / удалять** колонки;
- как **расширять тип** и эволюционировать вложенные структуры;
- почему это безопасно — колонки по **ID**;
- **снимки** (snapshots) и **time travel**;
- **эволюцию партиций**.

Та же таблица `user_events` в `bench.lakehouse`, что и в гайде 3.

## Подключение к Trino

Ноутбук ходит в Trino через Python-клиент `trino`. Если пакеты не установлены —
раскомментируй строку установки в ячейке ниже.

Параметры подключения задаются переменными:
- `TRINO_HOST` — `localhost`, если JupyterHub на **той же машине**, что и Trino;
  `host.docker.internal`, если JupyterHub в Docker; либо служебное имя Trino в вашем кластере.
- `CATALOG` / `SCHEMA` — каталог `bench` и схема `lakehouse` из гайда 3.

In [ ]:
# %pip install trino pandas    # раскомментируй, если пакетов нет в окружении

import trino
import pandas as pd

TRINO_HOST = "localhost"   # localhost / host.docker.internal / имя сервиса Trino
TRINO_PORT = 8080
TRINO_USER = "trino"
CATALOG = "bench"          # каталог из гайда 3
SCHEMA  = "lakehouse"      # схема, в которой лежат таблицы

conn = trino.dbapi.connect(
    host=TRINO_HOST,
    port=TRINO_PORT,
    user=TRINO_USER,
    catalog=CATALOG,
    schema=SCHEMA,
)

def q(sql):
    """SELECT -> pandas.DataFrame (наглядно в таблице)."""
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    return pd.DataFrame(rows, columns=cols)

def run(sql):
    """DDL/DML (CREATE/ALTER/INSERT/DROP) — выполняет, возвращает результат, если есть."""
    cur = conn.cursor()
    cur.execute(sql)
    try:
        return cur.fetchall()
    except Exception:
        return None

# проверка связи: должны увидеть каталоги, в т.ч. bench
q("SHOW CATALOGS")

## Типы данных в Iceberg

| Категория | Типы | Примечание |
|---|---|---|
| Логический | `boolean` | true / false |
| Целые | `integer`, `bigint` | 32 / 64 бита |
| Дробные | `double`, `decimal(P, S)` | `decimal` — точная арифметика денег |
| Строки | `varchar` | `string` из Spark в Trino называется `varchar` |
| Время | `timestamp` | в Trino ещё есть `timestamp(p)` без таймзоны |
| Сложные | `array(...)`, `map(k, v)`, `row(...)` | `struct` из Spark → `row` в Trino |

In [ ]:
# пересоздаём таблицу начисто, чтобы ноутбук можно было запускать повторно
run("DROP TABLE IF EXISTS bench.lakehouse.user_events")

run("""
CREATE TABLE bench.lakehouse.user_events (
    event_id     bigint,
    user_id      bigint,
    event_type   varchar,
    retry_count  integer,
    is_active    boolean,
    score        double,
    price        decimal(10, 2),
    created_at   timestamp,
    tags         array(varchar),
    attrs        map(varchar, varchar),
    meta         row(device varchar, os varchar)
) WITH (format = 'PARQUET', partitioning = ARRAY['event_type'])
""")

q("SHOW COLUMNS FROM bench.lakehouse.user_events")

В таблице есть и примитивы (`bigint`, `varchar`, `boolean`, `double`, `decimal`, `timestamp`),
и сложные типы (`array`, `map`, `row`). Наполним её данными.

Синтаксис сложных типов в Trino: массив — `ARRAY[...]`, словарь — `MAP(ARRAY[ключи], ARRAY[значения])`,
структура — `CAST(ROW(...) AS ROW(...))` (cast задаёт имена полей).

In [ ]:
run("""
INSERT INTO bench.lakehouse.user_events VALUES
(1, 100, 'click', 0, true, 3.5, 9.99, TIMESTAMP '2026-01-01 10:00:00',
 ARRAY['mobile', 'web'], MAP(ARRAY['lang'], ARRAY['ru']),
 CAST(ROW('iphone', 'ios') AS ROW(device varchar, os varchar))),
(2, 101, 'view',  1, true, 1.0, 0.00, TIMESTAMP '2026-01-02 10:00:00',
 ARRAY['desktop'], MAP(ARRAY['lang'], ARRAY['en']),
 CAST(ROW('pixel', 'android') AS ROW(device varchar, os varchar))),
(3, 102, 'click', 2, false, 5.0, 19.99, TIMESTAMP '2026-01-03 10:00:00',
 ARRAY['mobile'], MAP(ARRAY['lang'], ARRAY['ru']),
 CAST(ROW('galaxy', 'android') AS ROW(device varchar, os varchar)))
""")

q("SELECT * FROM bench.lakehouse.user_events ORDER BY event_id")

## 1. ADD COLUMN — добавить колонку (безопасно)

Новая колонка появляется **только в метаданных**. В уже записанных файлах её нет,
поэтому старые строки читают значение **NULL**. Перезаписи данных нет — операция мгновенная.

In [ ]:
run("ALTER TABLE bench.lakehouse.user_events ADD COLUMN browser_version varchar")

# новая запись уже несёт значение колонки
run("""
INSERT INTO bench.lakehouse.user_events VALUES
(4, 103, 'click', 0, true, 2.5, 4.99, TIMESTAMP '2026-01-04 10:00:00',
 ARRAY['mobile'], MAP(ARRAY['lang'], ARRAY['ru']),
 CAST(ROW('iphone', 'ios') AS ROW(device varchar, os varchar)), '17.4')
""")

# старые строки (1-3) видят NULL, новая (4) — '17.4'
q("SELECT event_id, browser_version FROM bench.lakehouse.user_events ORDER BY event_id")

## 2. RENAME COLUMN — переименовать колонку

Iceberg отслеживает каждую колонку по **уникальному ID**, а не по имени и не по позиции.
Поэтому rename — это правка только отображаемого имени в метаданных: старые файлы читаются
корректно, потому что ссылаются на тот же ID. Партиционный столбец тоже переименуется —
это та же колонка.

In [ ]:
run("ALTER TABLE bench.lakehouse.user_events RENAME COLUMN event_type TO action_category")

q("SELECT event_id, action_category FROM bench.lakehouse.user_events ORDER BY event_id")

## 3. DROP COLUMN — удалить колонку

Колонка исчезает из схемы — читатели её больше не видят. Сами данные **физически остаются**
в файлах (вернёмся к ним в разделе про снимки). Drop не «сдвигает» соседние колонки.

In [ ]:
run("ALTER TABLE bench.lakehouse.user_events DROP COLUMN browser_version")

q("SELECT * FROM bench.lakehouse.user_events ORDER BY event_id")
# колонки browser_version больше нет

## 4. Расширение типа (widen) — integer → bigint

Расширяющее изменение (каждое старое значение представимо в новом типе) безопасно:
старые данные на диске **не переписываются**, при чтении значение автоматически «повышается».

Синтаксис в Trino — `ALTER COLUMN ... SET DATA TYPE`. Сужение (`bigint` → `integer`) —
**ломающее**: Iceberg не даст его сделать без перезаписи данных.

In [ ]:
run("ALTER TABLE bench.lakehouse.user_events ALTER COLUMN retry_count SET DATA TYPE bigint")

q("SHOW COLUMNS FROM bench.lakehouse.user_events")
# retry_count теперь bigint (был integer)

## 5. Вложенные типы — поле внутри row/struct

Можно эволюционировать и вложенные структуры: добавлять поля в `row`, элементы в `array`/`map`.
Существующие записи получают **NULL** в новом поле.

> ⚠️ Перестановка колонок (`ALTER COLUMN ... FIRST` / `AFTER`) в Trino **не поддерживается**
> (есть в Spark, но это чисто косметика — данные не затрагиваются).

In [ ]:
run("ALTER TABLE bench.lakehouse.user_events ADD COLUMN meta.app_version varchar")

q("SELECT event_id, meta FROM bench.lakehouse.user_events ORDER BY event_id")
# у старых строк meta.app_version = NULL

## 6. Заглянем «под капот»: колонки по ID

В метаданных Iceberg каждая колонка хранится вместе со своим числовым **ID**.
Именно поэтому `rename` / `drop` не ломают старые файлы и не «сдвигают» соседей —
файлы ссылаются на ID, а не на имя или позицию:

- переименование — смена отображаемого имени у того же ID;
- удаление — пометка ID «невидимым», сами данные в файлах остаются.

Сами метаданные (с ID колонок) лежат в MinIO — бакет `bench`, папка
`user_events-…/metadata/*.json` (те же файлы, что ты видел в гайде 1). Через SQL ID напрямую
не показать, но эффект «по ID» виден выше: после `rename` старые данные читаются под новым именем.

> 📌 Не полагайся на позицию колонки — обращайся к ней по имени.

## 7. Снимки и time travel

Каждое **изменение данных** (INSERT/UPDATE/DELETE/MERGE) создаёт новый **снимок** (snapshot) —
ссылку на набор файлов, актуальных в этот момент. Благодаря снимкам таблицу можно читать
«на любой момент в прошлом».

> ⚠️ **Изменение схемы снимок НЕ создаёт.** `ADD/DROP/RENAME COLUMN`, `SET DATA TYPE` — это
> только новая версия метаданных. Снимки создают только операции с данными.

In [ ]:
# все снимки таблицы
q('''SELECT snapshot_id, operation, committed_at, summary['total-records'] AS total_records FROM bench.lakehouse."user_events$snapshots" ORDER BY committed_at''')

In [ ]:
snaps = q('SELECT snapshot_id, operation FROM bench.lakehouse."user_events$snapshots" ORDER BY committed_at')

# снимки: 0 — пустой (CREATE TABLE, total-records=0), 1 — первый INSERT (3 строки, до изменений схемы), 2 — второй INSERT (1 строка)
first_data_snap = snaps['snapshot_id'].iloc[1]
print("снимок «до изменений схемы»:", first_data_snap)

# читаем таблицу на тот момент: снова исходное имя event_type и исходный integer у retry_count
q(f"SELECT event_id, event_type, retry_count FROM bench.lakehouse.user_events FOR VERSION AS OF {first_data_snap} ORDER BY event_id")

## 8. Эволюция партиций

В классических форматах (Hive) смена партиции требовала **полной перезаписи** таблицы.
В Iceberg партицию можно заменить — это тоже операция над метаданными, файлы не трогаются.
Старые партиции продолжают читаться рядом с новыми.

В Trino партиция задаётся свойством `partitioning` и **заменяется целиком** одной командой
(в отличие от Spark, где поля добавляют/удаляют по одному).

In [ ]:
before = q("SELECT count(*) AS n FROM bench.lakehouse.user_events").iloc[0, 0]

# заменяем партицию: вместо текстовой колонки action_category — bucket(16, user_id)
run("ALTER TABLE bench.lakehouse.user_events SET PROPERTIES partitioning = ARRAY['bucket(user_id, 16)']")

after = q("SELECT count(*) AS n FROM bench.lakehouse.user_events").iloc[0, 0]
print(f"строк до/после смены партиции: {before} / {after} — данные не переписывались")

# видны и старые партиции (по action_category), и новая (bucket по user_id)
q('SELECT * FROM bench.lakehouse."user_events$partitions"')

## Шпаргалка: что безопасно, что ломает

| Операция | Оценка | Почему |
|---|---|---|
| `ADD COLUMN` | ✅ безопасно | NULL для старых данных, файлы не трогаются |
| `ADD COLUMN` в `row` | ✅ безопасно | новое поле структуры = NULL у старых строк |
| `widen` (integer→bigint, double→decimal) | ✅ безопасно | старое значение представимо в новом типе |
| `RENAME COLUMN` | ⚠️ безопасно в БД, рискованно операционно | старые читатели ищут колонку по имени |
| `DROP COLUMN` | ⚠️ осторожно | колонка исчезает; вернуть — обратным `ADD` (данные в файлах остаются, но колонку Iceberg создаст с новым ID) |
| сужение (bigint→integer, смена масштаба decimal) | ❌ ломает | значение может не представиться — нужна перезапись |
| reorder (`FIRST`/`AFTER`) | — | в Trino не поддерживается (есть в Spark; косметика) |

## Итог

Iceberg делает эволюцию схемы **мгновенной и безопасной** благодаря двум вещам:

1. **Колонки по ID** — операции адресуют колонку по идентификатору, а не по имени/позиции.
2. **Метаданные отдельно от данных** — изменение схемы — это новая версия метаданных, файлы не переписываются.

Эволюция схемы — это **операция над метаданными**, а не миграция данных.

Чтобы начать заново, выполни ячейку ниже.

In [ ]:
# Очистить таблицу, чтобы прогнать ноутбук с чистого листа:
# run("DROP TABLE bench.lakehouse.user_events")